# Beca 18 RAG Chatbot
Retrieval-Augmented Generation pipeline to answer questions about Peru's Beca 18 scholarship regulations (PRONABEC).

## Step 0 — Environment Setup
Load the Gemini API key from `.env` and print all relevant package versions.

In [1]:
import subprocess
subprocess.run(["pip", "install", "-r", "../requirements.txt", "-q"], check=True)

import os
from dotenv import load_dotenv, find_dotenv
from importlib.metadata import version

load_dotenv(find_dotenv())

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
print("API key loaded successfully.")

# Print package versions
import pypdf, tiktoken, chromadb, ipywidgets, tqdm
import google.genai as genai

packages = {
    "pypdf": pypdf.__version__,
    "tiktoken": tiktoken.__version__,
    "langchain-text-splitters": version("langchain-text-splitters"),
    "google-genai": genai.__version__,
    "chromadb": chromadb.__version__,
    "ipywidgets": ipywidgets.__version__,
    "tqdm": tqdm.__version__,
}

print("\nPackage versions:")
for pkg, ver in packages.items():
    print(f"  {pkg}: {ver}")

API key loaded successfully.

Package versions:
  pypdf: 6.11.0
  tiktoken: 0.13.0
  langchain-text-splitters: 1.1.2
  google-genai: 2.3.0
  chromadb: 1.5.9
  ipywidgets: 8.1.8
  tqdm: 4.67.1


## Step 1 — PDF Text Extraction
Extract text page-by-page using `pypdf`, insert `[PAGE N]` markers, and clean the raw text.

In [2]:
import re
import pypdf

PDF_PATH = "../data/beca18_reglamento.pdf"

def extract_text(pdf_path: str) -> str:
    reader = pypdf.PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        raw = page.extract_text() or ""
        # Insert page marker
        text = f"[PAGE {i}]\n{raw}"
        pages.append(text)
    return "\n".join(pages)

def clean_text(text: str) -> str:
    # Remove line breaks inside sentences (keep paragraph breaks)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    # Collapse multiple spaces
    text = re.sub(r" {2,}", " ", text)
    # Strip leading/trailing whitespace per line
    text = "\n".join(line.strip() for line in text.splitlines())
    # Collapse multiple blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

raw_text = extract_text(PDF_PATH)
full_text = clean_text(raw_text)

char_count = len(full_text)
word_count = len(full_text.split())

print(f"Characters : {char_count:,}")
print(f"Words      : {word_count:,}")
print(f"\n--- Preview (first 500 chars) ---\n{full_text[:500]}")

Characters : 366,272
Words      : 55,202

--- Preview (first 500 chars) ---
[PAGE 1] Resolución Directoral Ejecutiva Nº 033-2026-MINEDU/VMGI-PRONABEC Lima, 24 de febrero de 2026 VISTOS: El Informe N° 451-2026-MINEDU/VMGI-PRONABEC-DIBEC-SES, suscrito por la Dirección de Gestión de Becas y la Dirección de Acompañamiento Socioemocional y Bienestar; el Informe N° 042-2026-MINEDU/VMGI-PRONABEC-OPP de la Oficina de Planeamiento y Presupuesto; el Informe N ° 048-2026-MINEDU/VMGI-PRONABEC-OAJ de la Oficina de Asesoría Jurídica, y; CONSIDERANDO: Que, la Ley N° 29837 crea el Prog


## Step 2 — Tokenization & Chunking
Count total tokens with `tiktoken`, justify chunk parameters, and split the text with `RecursiveCharacterTextSplitter`.

In [3]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Token count ---
enc = tiktoken.get_encoding("cl100k_base")
total_tokens = len(enc.encode(full_text))
print(f"Total tokens in document: {total_tokens:,}")

# --- Chunk size justification ---
# Embedding limit: 8,192 tokens (gemini-embedding-001)
# chunk_size = 400 tokens  → well within the limit, captures coherent article/paragraph
# chunk_overlap = 60 tokens → ~15% overlap preserves context across chunk boundaries
print("\nChunk size  : 400 tokens  (well within 8,192-token embedding limit)")
print("Chunk overlap: 60 tokens  (~15% overlap to preserve cross-boundary context)")

# --- Splitter ---
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 60

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " "],
    length_function=lambda t: len(enc.encode(t)),
)

metadata_base = {
    "document": "beca18_reglamento",
    "topic": "beca18",
    "language": "es",
}

chunks = splitter.create_documents(
    texts=[full_text],
    metadatas=[metadata_base],
)

avg_len = sum(len(enc.encode(c.page_content)) for c in chunks) / len(chunks)
print(f"\nTotal chunks  : {len(chunks):,}")
print(f"Average length: {avg_len:.1f} tokens")
print(f"\n--- Sample chunk ---\n{chunks[0].page_content[:300]}")

Total tokens in document: 102,824

Chunk size  : 400 tokens  (well within 8,192-token embedding limit)
Chunk overlap: 60 tokens  (~15% overlap to preserve cross-boundary context)

Total chunks  : 321
Average length: 344.0 tokens

--- Sample chunk ---
[PAGE 1] Resolución Directoral Ejecutiva Nº 033-2026-MINEDU/VMGI-PRONABEC Lima, 24 de febrero de 2026 VISTOS: El Informe N° 451-2026-MINEDU/VMGI-PRONABEC-DIBEC-SES, suscrito por la Dirección de Gestión de Becas y la Dirección de Acompañamiento Socioemocional y Bienestar; el Informe N° 042-2026-MINED


## Step 3 — Embedding Functions
Implement `embed_documents` and `embed_query` using `gemini-embedding-001` (768 dimensions) with exponential backoff for rate limiting.

In [4]:
import time
import google.genai as genai

client = genai.Client(api_key=GEMINI_API_KEY)

EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIMS  = 768

def _embed_with_backoff(texts: list[str], task_type: str) -> list[list[float]]:
    max_retries = 5
    for attempt in range(max_retries):
        try:
            response = client.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=texts,
                config={"task_type": task_type, "output_dimensionality": EMBEDDING_DIMS},
            )
            return [e.values for e in response.embeddings]
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt
            print(f"Rate limit hit, retrying in {wait}s... ({e})")
            time.sleep(wait)

def embed_documents(texts: list[str]) -> list[list[float]]:
    return _embed_with_backoff(texts, "RETRIEVAL_DOCUMENT")

def embed_query(text: str) -> list[float]:
    return _embed_with_backoff([text], "RETRIEVAL_QUERY")[0]

# Smoke test
test_embedding = embed_query("¿Quiénes pueden postular a Beca 18?")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Embedding dims  : {len(test_embedding)}")
print(f"Sample values   : {test_embedding[:5]}")

Rate limit hit, retrying in 1s... (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
Rate limit hit, retrying in 2s... (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
Rate limit hit, retrying in 4s... (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
Rate limit hit, ret

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}}

## Step 4 — ChromaDB Indexing
Create a persistent ChromaDB collection with cosine distance and index all chunks idempotently.

In [5]:
import chromadb
from tqdm import tqdm

# Absolute path relative to this notebook's location
NOTEBOOK_DIR     = os.path.dirname(os.path.abspath("beca18_rag_chatbot.ipynb"))
CHROMA_PATH      = os.path.join(NOTEBOOK_DIR, "..", "chroma_db_beca18")
COLLECTION_NAME  = "beca18_reglamento"
BATCH_SIZE       = 10

print(f"ChromaDB path: {os.path.abspath(CHROMA_PATH)}")

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

print(f"Documents currently in collection: {collection.count()}")

if collection.count() == 0:
    print(f"Indexing {len(chunks)} chunks into ChromaDB...")
    for i in tqdm(range(0, len(chunks), BATCH_SIZE)):
        batch      = chunks[i : i + BATCH_SIZE]
        texts      = [c.page_content for c in batch]
        metadatas  = [c.metadata for c in batch]
        ids        = [f"chunk_{i + j}" for j in range(len(batch))]
        embeddings = embed_documents(texts)
        collection.add(documents=texts, embeddings=embeddings, metadatas=metadatas, ids=ids)
        time.sleep(5)
    print("Indexing complete.")
else:
    print("Collection already populated — skipping indexing.")

print(f"\nTotal documents stored: {collection.count()}")

ChromaDB path: /home/nico/Documents/GitHub/beca18-rag-chatbot/chroma_db_beca18
Documents currently in collection: 0
Indexing 321 chunks into ChromaDB...


100%|██████████| 33/33 [03:37<00:00,  6.58s/it]

Indexing complete.

Total documents stored: 321


## Step 5 — Semantic Search
Build `semantic_search` using the embedded query to retrieve the top-k most relevant chunks from ChromaDB.

In [8]:
def semantic_search(question: str, k: int = 5) -> list[dict]:
    query_embedding = embed_query(question)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {
            "text":     results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
        }
        for i in range(len(results["documents"][0]))
    ]

# Test with one sample question — print top 3
sample_question = "¿Cuáles son los requisitos para postular a Beca 18?"
top_results = semantic_search(sample_question, k=5)

print(f"Query: {sample_question}\n")
for i, r in enumerate(top_results[:3], start=1):
    print(f"--- Result {i} ---")
    print(f"Distance : {r['distance']:.4f}")
    print(f"Metadata : {r['metadata']}")
    print(f"Text     : {r['text'][:300]}\n")

Rate limit hit, retrying in 1s... (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
Rate limit hit, retrying in 2s... (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
Rate limit hit, retrying in 4s... (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
Rate limit hit, ret

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}}

## Step 6 — RAG Answer Generation
Implement `answer_with_context` using `gemini-2.5-flash`. Test with 5 on-topic questions and 1 off-topic question.

In [7]:
LLM_MODEL = "gemini-2.5-flash"

SYSTEM_PROMPT = """Eres un asistente experto en el Reglamento de Beca 18 de PRONABEC (Perú).
Responde ÚNICAMENTE basándote en los fragmentos del documento proporcionados como contexto.
Cuando la información esté disponible, cita el número de página usando el formato [PAGE N].
Si el contexto no contiene información suficiente para responder la pregunta, responde exactamente:
"El documento no contiene información sobre este tema."
No inventes ni inferras información que no esté explícitamente en el contexto."""

def answer_with_context(question: str, k: int = 5) -> str:
    results = semantic_search(question, k=k)
    context = "\n\n".join(
        f"[Fragmento {i+1}]\n{r['text']}" for i, r in enumerate(results)
    )
    prompt = f"""Contexto del reglamento:\n{context}\n\nPregunta: {question}"""
    response = client.models.generate_content(
        model=LLM_MODEL,
        contents=prompt,
        config={"system_instruction": SYSTEM_PROMPT},
    )
    return response.text

# --- 5 on-topic questions ---
questions = [
    ("Elegibilidad",   "¿Quiénes pueden postular a Beca 18?"),
    ("Modalidades",    "¿Cuáles son las modalidades de Beca 18?"),
    ("Estipendio",     "¿Qué beneficios económicos otorga Beca 18?"),
    ("Obligaciones",   "¿Cuáles son las obligaciones del becario?"),
    ("Pérdida de beca","¿En qué casos se pierde la beca 18?"),
    ("Fuera de tema",  "¿Cuál es la capital de Francia?"),
]

for topic, q in questions:
    print(f"{'='*60}")
    print(f"[{topic}] {q}")
    print(f"{'='*60}")
    print(answer_with_context(q))
    print()

[Elegibilidad] ¿Quiénes pueden postular a Beca 18?


NameError: name 'semantic_search' is not defined

## Step 7 — Interactive Chat UI
Build an interactive chatbot using `ipywidgets` with a text input, Ask/Clear buttons, k-slider, answer area, and expandable source accordion.

In [6]:
import ipywidgets as widgets
from IPython.display import display, HTML

# --- Widgets ---
question_input = widgets.Text(
    placeholder="Escribe tu pregunta sobre Beca 18...",
    layout=widgets.Layout(width="70%"),
)
k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description="k (chunks):",
    style={"description_width": "initial"},
)
ask_button   = widgets.Button(description="Ask",   button_style="primary")
clear_button = widgets.Button(description="Clear", button_style="warning")
answer_out   = widgets.Output()
sources_acc  = widgets.Accordion(children=[], titles=[])

def on_ask(b):
    question = question_input.value.strip()
    if not question:
        return
    k = k_slider.value

    answer_out.clear_output()
    sources_acc.children = []

    with answer_out:
        print("Buscando respuesta...")

    results  = semantic_search(question, k=k)
    answer   = answer_with_context(question, k=k)

    answer_out.clear_output()
    with answer_out:
        display(HTML(f"<b>Pregunta:</b> {question}"))
        display(HTML(f"<b>Respuesta:</b><br>{answer.replace(chr(10), '<br>')}"))

    source_widgets = []
    for i, r in enumerate(results):
        page = r["metadata"].get("page", "?")
        dist = r["distance"]
        content = widgets.HTML(
            value=(
                f"<b>Página:</b> {page} &nbsp;|&nbsp; <b>Distancia:</b> {dist:.4f}<br><br>"
                f"<pre style='white-space:pre-wrap'>{r['text'][:600]}</pre>"
            )
        )
        source_widgets.append(content)

    sources_acc.children = tuple(source_widgets)
    for i in range(len(source_widgets)):
        sources_acc.set_title(i, f"Fragmento {i+1}  (dist: {results[i]['distance']:.4f})")

def on_clear(b):
    question_input.value = ""
    answer_out.clear_output()
    sources_acc.children = []

ask_button.on_click(on_ask)
clear_button.on_click(on_clear)

# --- Layout ---
toolbar = widgets.HBox([question_input, ask_button, clear_button])
ui      = widgets.VBox([
    widgets.HTML("<h3>Chatbot Beca 18 — RAG</h3>"),
    toolbar,
    k_slider,
    widgets.HTML("<hr><b>Respuesta:</b>"),
    answer_out,
    widgets.HTML("<b>Fuentes recuperadas:</b>"),
    sources_acc,
])

display(ui)

NameError: name 'semantic_search' is not defined